# WebSocket mode for the Azure OpenAI Responses API

This notebook follows the [Responses API WebSocket examples from Microsoft Learn](https://learn.microsoft.com/azure/foundry/openai/how-to/websockets?branch=main&tabs=api-key). It opens one persistent connection to `/openai/v1/responses`, sends `response.create` events, streams server events, and continues later turns with `previous_response_id`.

WebSocket mode is useful for long-running, tool-heavy workflows because the socket stays open and follow-up turns only send incremental input. It is compatible with `store=false` and Zero Data Retention scenarios.

## Prerequisites

Before running the notebook, make sure you have:

- An Azure OpenAI model deployment.
- A `.env` file with `AZURE_OPENAI_V1_API_ENDPOINT`, `AZURE_OPENAI_API_MODEL`, and either `AZURE_OPENAI_API_KEY` or Microsoft Entra ID access.
- The `websocket-client` package installed.
- The `azure-identity` package installed if you use Microsoft Entra ID authentication.

## When to use WebSocket mode

Use WebSocket mode when a workflow has many model-tool round trips, such as agentic coding or orchestration loops. For single-shot requests or short conversations, the standard HTTP Responses API is usually simpler.

## Setup

The existing samples in this repository use `.env` variables. This cell loads those variables and derives the WebSocket URL from the Azure OpenAI v1 endpoint.

In [45]:
import json
import os
from urllib.parse import urlparse, urlunparse

from dotenv import load_dotenv
from websocket import create_connection

load_dotenv()

endpoint = os.environ["AZURE_OPENAI_V1_API_ENDPOINT"].rstrip("/")
model = os.environ["AZURE_OPENAI_API_MODEL"]

parsed_endpoint = urlparse(endpoint)
if parsed_endpoint.scheme not in {"http", "https"}:
    raise ValueError("AZURE_OPENAI_V1_API_ENDPOINT must start with http:// or https://.")

ws_scheme = "wss" if parsed_endpoint.scheme == "https" else "ws"
responses_ws_url = urlunparse(parsed_endpoint._replace(scheme=ws_scheme, path=f"{parsed_endpoint.path.rstrip('/')}/responses"))

print(f"Model deployment: {model}")
print(f"Responses WebSocket URL: {responses_ws_url}")

Model deployment: gpt-5.4
Responses WebSocket URL: wss://sweden-fp-resource.openai.azure.com/openai/v1/responses


## Authentication

The WebSocket request uses an `Authorization: Bearer ...` header. By default this notebook uses `AZURE_OPENAI_API_KEY`. Set `USE_ENTRA_ID = True` in the next cell to use Microsoft Entra ID instead.

In [46]:
USE_ENTRA_ID = True

def get_auth_header():
    if USE_ENTRA_ID:
        from azure.identity import DefaultAzureCredential

        credential = DefaultAzureCredential()
        token = credential.get_token("https://cognitiveservices.azure.com/.default").token
        return f"Authorization: Bearer {token}"

    api_key = os.getenv("AZURE_OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Set AZURE_OPENAI_API_KEY or set USE_ENTRA_ID = True in this cell.")
    return f"Authorization: Bearer {api_key}"


auth_header = get_auth_header()
print("Authentication header prepared.")

Authentication header prepared.


## Open the WebSocket connection

Open one socket to `/v1/responses`. The following cells reuse this same connection until the cleanup step closes it.

In [47]:
ws = create_connection(responses_ws_url, header=[auth_header])
print("WebSocket connection opened.")

WebSocket connection opened.


## Stream server events

Server events use the same event model as Responses API streaming. This helper prints text deltas as they arrive, returns the completed response ID, and raises an exception for error events.

In [48]:
def receive_response(ws):
    response_id = None

    while True:
        event = json.loads(ws.recv())
        event_type = event.get("type")

        if event_type == "response.output_text.delta":
            print(event.get("delta", ""), end="", flush=True)

        elif event_type == "response.completed":
            response_id = event.get("response", {}).get("id")
            print(f"\nResponse ID: {response_id}")
            return response_id

        elif event_type == "error":
            raise RuntimeError(json.dumps(event, indent=2))

        elif event_type == "response.failed":
            raise RuntimeError(json.dumps(event, indent=2))

## Start a turn

Send a `response.create` event on the open socket. This first turn mirrors the HTTP create body, except transport-specific fields like `stream` and `background` do not apply.

In [49]:
initial_request = {
    "type": "response.create",
    "model": model,
    "store": False,
    "input": [
        {
            "type": "message",
            "role": "user",
            "content": [{"type": "input_text", "text": "Tell me a joke."}],
        }
    ],
    "tools": [],
}

ws.send(json.dumps(initial_request))
response_id = receive_response(ws)

Why don’t skeletons fight each other?

Because they don’t have the guts.

If you want, I can tell you a drier one, a darker one, or a very bad pun.
Response ID: resp_080210f9eb9bda730169f29bcfc97c81969b2e809541d38d74


## Continue with incremental inputs

To continue the same chain, send another `response.create` on the same socket with `previous_response_id` set to the prior response ID. The `input` array contains only new items.

In [50]:
follow_up_request = {
    "type": "response.create",
    "model": model,
    "store": False,
    "previous_response_id": response_id,
    "input": [
        {
            "type": "message",
            "role": "user",
            "content": [{"type": "input_text", "text": "Explain why your joke was funny."}],
        }
    ],
    "tools": [],
}

ws.send(json.dumps(follow_up_request))
response_id = receive_response(ws)

It’s funny because it plays on a double meaning of “guts.”

- Literally, skeletons don’t have internal organs, so they truly don’t have guts.
- Figuratively, “having guts” means having courage.

So the joke sets you up to think the reason is about bravery, but the punchline also works as a literal fact about skeletons. That mismatch is the humor.
Response ID: resp_080210f9eb9bda730169f29bd1c4c0819689db16b139a6aca9


## Optional warmup with `generate: false`

You can optionally warm up request state by sending `response.create` with `generate: false`. A warmup does not return model output, but it does return a response ID and becomes the most recent response in the WebSocket's connection-local cache. The next generated turn must chain from that warmup response ID, otherwise the prior chain will not be found in the cache.

In [51]:
RUN_OPTIONAL_WARMUP = True

if RUN_OPTIONAL_WARMUP:
    warmup_request = {
        "type": "response.create",
        "model": model,
        "store": False,
        "generate": False,
        "previous_response_id": response_id,
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": [{"type": "input_text", "text": "Prepare to tell a short joke and explain why it works."}],
            }
        ],
        "tools": [],
    }

    ws.send(json.dumps(warmup_request))
    warmup_response_id = receive_response(ws)

    # The warmup is now the most recent response in the WebSocket cache.
    # Chain subsequent turns from this response ID.
    response_id = warmup_response_id
else:
    print("Skipped optional warmup. Set RUN_OPTIONAL_WARMUP = True to run this cell.")


Response ID: resp_080210f9eb9bda730169f29bd3f9dc8196a4af83dc1b5abbf9


## Continuation semantics

On an active WebSocket connection, the service keeps the most recent previous response in a connection-local in-memory cache. Continuing from that response is fast and works with `store=false` and ZDR because the state is not written to disk.

If a `previous_response_id` is not in the in-memory cache, behavior depends on `store`: with `store=true`, the service may hydrate older response IDs from persisted state; with `store=false`, there is no persisted fallback and the request returns `previous_response_not_found`. If a turn fails with a `4xx` or `5xx`, the service evicts the referenced previous response ID from the connection-local cache.

## Optional server-side compaction

When you enable server-side compaction with `context_management`, compaction happens during normal `/responses` generation. In WebSocket mode, continue the same way: send the next `response.create` with the latest `previous_response_id` and only new input items.

In [52]:
RUN_OPTIONAL_SERVER_SIDE_COMPACTION = True

if RUN_OPTIONAL_SERVER_SIDE_COMPACTION:
    compacting_request = {
        "type": "response.create",
        "model": model,
        "store": False,
        "previous_response_id": response_id,
        "context_management": [{"type": "compaction", "compact_threshold": 200000}],
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": [{"type": "input_text", "text": "Explain the joke in one sentence."}],
            }
        ],
        "tools": [],
    }

    ws.send(json.dumps(compacting_request))
    response_id = receive_response(ws)
else:
    print("Skipped optional server-side compaction. Set RUN_OPTIONAL_SERVER_SIDE_COMPACTION = True to run this cell.")

The joke is funny because “guts” means both courage and internal organs, so the punchline works both figuratively and literally for skeletons.
Response ID: resp_080210f9eb9bda730169f29bd4422c819689976281eef5a022


## Optional standalone `/responses/compact`

The standalone compact endpoint returns a new compacted input window, not a response ID. After compaction, start a new WebSocket response by omitting `previous_response_id` and passing the compacted output as input, plus the next user or tool items.

In [53]:
RUN_OPTIONAL_STANDALONE_COMPACTION = True

if RUN_OPTIONAL_STANDALONE_COMPACTION:
    from openai import OpenAI

    client = OpenAI(api_key=os.getenv("AZURE_OPENAI_API_KEY"), base_url=endpoint)
    long_input_items = [
        {
            "type": "message",
            "role": "user",
            "content": "Tell me a joke.",
        },
        {
            "id": "msg_example_001",
            "type": "message",
            "status": "completed",
            "role": "assistant",
            "content": [
                {
                    "type": "output_text",
                    "annotations": [],
                    "logprobs": [],
                    "text": "Why did the developer go broke? Because they used up all their cache.",
                }
            ],
        },
    ]

    # Compact your current window (HTTP call)
    compacted = client.responses.compact(
        model=model,
        input=long_input_items,
    )

    # Start a new response on the WebSocket using the compacted window.
    # SDK output items are Pydantic models, so pass a default= serializer to json.dumps.
    ws.send(json.dumps(
        {
            "type": "response.create",
            "model": model,
            "store": False,
            "input": [
                *compacted.output,
                {
                    "type": "message",
                    "role": "user",
                    "content": [{"type": "input_text", "text": "Continue from here."}],
                },
            ],
            "tools": [],
        },
        default=lambda obj: obj.model_dump(exclude_none=True),
    ))

    response_id = receive_response(ws)
else:
    print("Skipped optional standalone compaction. Set RUN_OPTIONAL_STANDALONE_COMPACTION = True to run this cell.")

Sure — here’s another:

Why do programmers prefer dark mode?  
Because light attracts bugs.
Response ID: resp_0ac00ce72e4f956c0169f29bd7ae808196aa3f860534f6df4e


## Connection behavior and limits

A single WebSocket connection can receive multiple `response.create` messages, but it runs them sequentially: one in-flight response at a time. Multiplexing is not supported. Connections are limited to 60 minutes, so long-running workflows should reconnect before or when that limit is reached.

## Optional reconnect and recover

When a connection closes or reaches the 60-minute limit, open a new WebSocket connection. With `store=true`, you can continue from a persisted `previous_response_id`; with `store=false` or ZDR, start a fresh response with the full or compacted context.

In [54]:
RUN_OPTIONAL_RECONNECT = True

if RUN_OPTIONAL_RECONNECT:
    ws.close()
    ws = create_connection(responses_ws_url, header=[auth_header])

    recovery_request = {
        "type": "response.create",
        "model": model,
        "store": False,
        "input": [
            {
                "type": "message",
                "role": "user",
                "content": [{"type": "input_text", "text": "Start a fresh chain after reconnecting by telling a new joke."}],
            }
        ],
        "tools": [],
    }

    ws.send(json.dumps(recovery_request))
    response_id = receive_response(ws)
else:
    print("Skipped optional reconnect. Set RUN_OPTIONAL_RECONNECT = True to run this cell.")

Why don’t skeletons fight each other?

They don’t have the guts.
Response ID: resp_09f8231c71b9a5a60169f29bda19308193bd80320180bd6391


## Troubleshooting

- `previous_response_not_found`: the referenced response ID is not in the connection-local cache and there is no persisted state to hydrate from. Start a new chain, or use `store=true` if your scenario allows persisted state.
- `websocket_connection_limit_reached`: the connection has been open for 60 minutes. Open a new WebSocket connection and recover with either a persisted `previous_response_id` or a fresh full/compacted context.

## Cleanup

Close the socket only when you are finished with all turns.

In [55]:
try:
    ws.close()
    print("WebSocket connection closed.")
except NameError:
    print("No WebSocket connection was opened.")

WebSocket connection closed.
